<a href="https://colab.research.google.com/github/shridharch/support-ticket-triage-langchain/blob/main/notebooks/ticket_triage_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **#Support Ticket Triage with** **LangChain** **bold text**

### **"Setup: install packages"**

In [1]:
  !pip install -q langchain langchain-groq langgraph pydantic

In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "ticket-triage"

In [3]:
!pip install -q langchain langchain-google-genai langgraph pydantic

In [4]:
from groq import Groq
for m in Groq().models.list().data:
    print(m.id)

meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
canopylabs/orpheus-v1-english
allam-2-7b
qwen/qwen3.8-27b
whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi
whisper-large-v3
openai/gpt-oss-safeguard-20b


## **"Load the model"**

In [5]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai/gpt-oss-120b", model_provider="groq")
print(llm.invoke("Say hello in one sentence.").text)

Hello!


## **"Build the chain"**

In [6]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class Ticket(BaseModel):
    category: Literal["billing", "bug", "feature_request", "other"] = Field(description="Ticket category")
    urgency: Literal["low", "medium", "high"] = Field(description="Ticket urgency")
    summary: str = Field(description="One-sentence summary of the issue")

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You triage customer support emails.\n"
     "Categories: billing (charges, refunds, invoices), bug (something broken), "
     "feature_request (asking for new functionality), other.\n"
     "Urgency: high = outage, money lost, or angry customer; "
     "medium = blocked but has workaround; low = general question."),
    ("human", "{email}"),
])

chain = prompt | llm.with_structured_output(Ticket)

print(chain.invoke({"email": "My card was charged twice, fix this ASAP!"}))

category='billing' urgency='high' summary='Card was charged twice'


## **"Test on 10 emails"**

In [7]:
import time

emails = [
    "My card was charged twice, fix this ASAP!",
    "The app crashes every time I click Export.",
    "Could you add dark mode? It would be great.",
    "Site is down for all our customers!!",
    "How do I change my profile picture?",
    "I was billed $50 but my plan is $20.",
    "Login button doesn't work on Safari, using Chrome instead for now.",
    "Love the product, just saying thanks.",
    "Please add CSV export for reports.",
    "Refund me now or I'm cancelling and posting reviews everywhere!",
]

lines = []
for e in emails:
    try:
        r = chain.invoke({"email": e})
        lines.append(f"{r.category:16} {r.urgency:7} | {e}")
    except Exception as ex:
        lines.append(f"{'FAILED':16} {'-':7} | {e}  <- {type(ex).__name__}")
    time.sleep(3)

print("\n".join(lines))

billing          high    | My card was charged twice, fix this ASAP!
bug              high    | The app crashes every time I click Export.
feature_request  low     | Could you add dark mode? It would be great.
bug              high    | Site is down for all our customers!!
other            low     | How do I change my profile picture?
billing          high    | I was billed $50 but my plan is $20.
bug              medium  | Login button doesn't work on Safari, using Chrome instead for now.
other            low     | Love the product, just saying thanks.
feature_request  low     | Please add CSV export for reports.
billing          high    | Refund me now or I'm cancelling and posting reviews everywhere!


# **Next: Stage 2 (tools)**

### Define two **tools**

In [8]:
from typing import Literal
from langchain_core.tools import tool

@tool
def create_ticket(
    category: Literal["billing", "bug", "feature_request", "other"],
    urgency: Literal["low", "medium", "high"],
    summary: str,
) -> str:
    """Save a support ticket to the database."""
    print(f"[create_ticket] {category} | {urgency} | {summary}")
    return "Ticket #123 created"

@tool
def notify_team(message: str) -> str:
    """Send an urgent alert to the support team. Use only for high-urgency tickets."""
    print(f"[notify_team] {message}")
    return "Team notified"

**let the model choose tools**

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

llm_with_tools = llm.bind_tools([create_ticket, notify_team])
tools = {"create_ticket": create_ticket, "notify_team": notify_team}

system = SystemMessage(
    "You handle support emails.\n"
    "Step 1: call create_ticket with category (billing, bug, feature_request, other) "
    "and urgency (low, medium, high).\n"
    "Step 2: if urgency is high (outage, money lost, or angry customer), you MUST also "
    "call notify_team.\n"
    "Do not skip notify_team for high urgency."
)

def handle(email, max_rounds=3):
    messages = [system, HumanMessage(email)]
    print("EMAIL:", email)
    for _ in range(max_rounds):
        ai = llm_with_tools.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            break
        for call in ai.tool_calls:
            result = tools[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
    print("-" * 40)

## `Test it`

In [10]:
handle("Site is down for all our customers!!")
handle("How do I change my profile picture?")

EMAIL: Site is down for all our customers!!
[create_ticket] bug | high | Site is down for all customers
[notify_team] Urgent: Site is down for all customers. Immediate attention required.
----------------------------------------
EMAIL: How do I change my profile picture?
[create_ticket] other | low | User asks how to change profile picture.
----------------------------------------


## **Stage 3: LangGraph workflow**

### Connect classify, route, alert, and log steps into a graph where the routing decision is made in code, not by the model.

In [12]:
#Langraph workflow

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    email: str
    ticket: dict
    actions: list

def classify(state: State):
    t = chain.invoke({"email": state["email"]})
    return {"ticket": t.model_dump(), "actions": []}

def alert_team(state: State):
    out = notify_team.invoke({"message": state["ticket"]["summary"]})
    return {"actions": state["actions"] + [out]}

def log_ticket(state: State):
    out = create_ticket.invoke(state["ticket"])
    return {"actions": state["actions"] + [out]}

def route(state: State):
    return "alert_team" if state["ticket"]["urgency"] == "high" else "log_ticket"

g = StateGraph(State)
g.add_node("classify", classify)
g.add_node("alert_team", alert_team)
g.add_node("log_ticket", log_ticket)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route, ["alert_team", "log_ticket"])
g.add_edge("alert_team", "log_ticket")
g.add_edge("log_ticket", END)

app = g.compile()
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	alert_team(alert_team)
	log_ticket(log_ticket)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	alert_team --> log_ticket;
	classify -.-> alert_team;
	classify -.-> log_ticket;
	log_ticket --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [13]:
for email in ["Site is down for all our customers!!",
              "How do I change my profile picture?",
              "I was billed $50 but my plan is $20."]:
    result = app.invoke({"email": email})
    print(result["ticket"])
    print(result["actions"])
    print("-" * 40)

[notify_team] Site is down for all customers
[create_ticket] bug | high | Site is down for all customers
{'category': 'bug', 'urgency': 'high', 'summary': 'Site is down for all customers'}
['Team notified', 'Ticket #123 created']
----------------------------------------
[create_ticket] other | low | User asks how to change profile picture
{'category': 'other', 'urgency': 'low', 'summary': 'User asks how to change profile picture'}
['Ticket #123 created']
----------------------------------------
[notify_team] Charged $50 instead of $20 plan
[create_ticket] billing | high | Charged $50 instead of $20 plan
{'category': 'billing', 'urgency': 'high', 'summary': 'Charged $50 instead of $20 plan'}
['Team notified', 'Ticket #123 created']
----------------------------------------
